In [30]:
# activity data
import pandas as pd
import numpy as np
import os 
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import f1_score,confusion_matrix,precision_score,recall_score
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler

from imblearn.over_sampling import RandomOverSampler

import matplotlib.pyplot as plt
import seaborn as sns
import shap

from utils.data_utils import correct_col_type,gen_date_col,transform_category_to_counts,min_max_perpatient

In [31]:
# Please change the path with the path of your dataset
DPATH = '../Dataset/'

# Read all tables into data_dict

files = os.listdir(DPATH)
data_dict = {}
summaries = {}
for f in files:
    if 'csv' not in f:
        continue
    print(f)
    fpth = os.path.join(DPATH,f)
    df = pd.read_csv(fpth)

    for col in df.columns:
        df[col] = correct_col_type(df,col)
    if 'date' in df.columns:
        df = df.rename(columns={'date':'timestamp'})
                
    fname = f.split('.')[0]
    data_dict[fname] = df

In [32]:
sleep_df1 = gen_date_col(data_dict['Sleep'],tcol='timestamp')
sleep_df1

# The total minutes of sleep in the given 22:00 to 06:00 period (22:00-24:00 in the previous day, and 0:00-6:00 in the same day)

In [33]:
from datetime import time, timedelta
sleep_df1 = sleep_df1[
    (df['timestamp'].dt.time < time(6, 0)) | 
    (df['timestamp'].dt.time >= time(22, 0))
]

sleep_df1

In [34]:
sleep_id_array = sleep_df1.patient_id.unique()
sleep_id_array

In [35]:
from datetime import time, timedelta
# Define function for custom sleep_date
def get_sleep_date(ts):
    t = ts.time()
    if time(0, 0) <= t < time(22, 0):
        return ts.date()
    else:
        return (ts + timedelta(days=1)).date()

# Apply the function
sleep_df1['sleep_date'] = sleep_df1['timestamp'].apply(get_sleep_date)
sleep_df1

In [51]:
import pandas as pd
import datetime

summary_df_list =[]

for id_pick in sleep_id_array:
    print(id_pick)
    sleep_df_idPick = sleep_df1[sleep_df1['patient_id'] == id_pick] 
    # sleep date (follow the rule that 22:00-24:00 belongs to the next day)
    date_array = sleep_df_idPick.sleep_date.unique()
    SLEEP_duration_list = []
    LIGHT_duration_list = []
    DEEP_duration_list = []
    REM_duration_list = []
    date_list = []
    
    for target_date in date_array:
        date_list.append(target_date)
        # in each day
        filtered_df = sleep_df_idPick[sleep_df_idPick['sleep_date'] == target_date]
        
        # Assuming your dataframe is called sleep_df_idPick
        # First, make sure timestamp is a datetime object
        filtered_df['timestamp'] = pd.to_datetime(filtered_df['timestamp'])
        
        # Sort by timestamp
        filtered_df = filtered_df.sort_values('timestamp').reset_index(drop=True)
        
    ##################### calculate sleep duration ##########################
        # Identify chunks of consecutive same state
        filtered_df['state_change'] = (filtered_df['state'] != filtered_df['state'].shift()).cumsum()
        
        # Group by each chunk
        chunk_durations = filtered_df.groupby(['state_change', 'state']).agg(
            start_time=('timestamp', 'first'),
            end_time=('timestamp', 'last')
        ).reset_index()
        chunk_durations = chunk_durations.dropna(subset=['start_time']).reset_index()
        
        # Calculate duration in minutes per chunk
        chunk_durations['duration_min'] = (chunk_durations['end_time'] - chunk_durations['start_time']).dt.total_seconds() / 60
        
        # Now sum duration per state, and convert to hours
        state_total_duration = chunk_durations.groupby('state')['duration_min'].sum()   # in minutes
        # print(state_total_duration)
        SLEEP_duration = state_total_duration['LIGHT'] + state_total_duration['DEEP'] + state_total_duration['REM']   
        
        # append data
        SLEEP_duration_list.append(SLEEP_duration)
        LIGHT_duration_list.append(state_total_duration['LIGHT']) 
        DEEP_duration_list.append(state_total_duration['DEEP'])
        REM_duration_list.append(state_total_duration['REM'])
        
    sleep_summary = {
        'patient_id': id_pick,
        'sleep_date': date_list,
        'sleep_duration': SLEEP_duration_list,
        'light_duration': LIGHT_duration_list,
        'deep_duration': DEEP_duration_list,
        'REM_duration': REM_duration_list
        }

    sleep_summary_df = pd.DataFrame(sleep_summary)  
    summary_df_list.append(sleep_summary_df)


In [64]:
final_df = pd.concat(summary_df_list, ignore_index=True)
final_df.to_csv('../output/data_cleaned/sleep_night_durations_each_state_total.csv', index=False)
final_df